## Подключение библиотек

На первом этапе подключаются основные библиотеки, необходимые для анализа данных и построения модели машинного обучения.

- **pandas** — используется для загрузки и обработки табличных данных.
- **scikit-learn** — библиотека для машинного обучения:
  - `train_test_split` — разделение данных на обучающую и тестовую выборки.
  - `OrdinalEncoder` — кодирование категориальных признаков в числовой формат.
  - `RandomForestClassifier` — модель ансамблевого обучения (случайный лес).
  - `metrics` — вычисление метрик качества модели.

После подключения библиотек можно переходить к загрузке исходных данных.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

## Загрузка исходных данных

На данном этапе формируется список CSV-файлов, содержащих данные исследования.

Файлы загружаются и объединяются в единый DataFrame `df`.
Это позволяет работать со всей совокупностью наблюдений как с одной таблицей.

Результатом выполнения ячейки является общий датасет, содержащий информацию о женщинах и исходах беременности.

In [2]:
files = [
    "../archive/AHS_Woman_08_Rajasthan/AHS_Woman_08_Rajasthan.csv",
    "../archive/AHS_Woman_10_Bihar/AHS_Woman_10_Bihar_Part_1.csv",
    "../archive/AHS_Woman_10_Bihar/AHS_Woman_10_Bihar_Part_2.csv",
    "../archive/AHS_Woman_18_Assam/AHS_Woman_18_Assam.csv",
    "../archive/AHS_Woman_20_Jharkhand/AHS_Woman_20_Jharkhand.csv",
    "../archive/AHS_Woman_21_Odisha/AHS_Woman_21_Odisha.csv",
    "../archive/AHS_Woman_22_Chhattisgarh/AHS_Woman_22_Chhattisgarh.csv",
    "../archive/AHS_Woman_23_Madhya_Pradesh/AHS_Woman_23_Madhya_Pradesh_Part_1.csv",
    "../archive/AHS_Woman_23_Madhya_Pradesh/AHS_Woman_23_Madhya_Pradesh_Part_2.csv"
]

dfs = []

for f in files:
    print("Reading:", f)
    temp = pd.read_csv(f, sep="|", low_memory=False)
    dfs.append(temp)

df = pd.concat(dfs, ignore_index=True)

print("Data merged")
print("Dataset shape:", df.shape)

Reading: ../archive/AHS_Woman_08_Rajasthan/AHS_Woman_08_Rajasthan.csv
Reading: ../archive/AHS_Woman_10_Bihar/AHS_Woman_10_Bihar_Part_1.csv
Reading: ../archive/AHS_Woman_10_Bihar/AHS_Woman_10_Bihar_Part_2.csv
Reading: ../archive/AHS_Woman_18_Assam/AHS_Woman_18_Assam.csv
Reading: ../archive/AHS_Woman_20_Jharkhand/AHS_Woman_20_Jharkhand.csv
Reading: ../archive/AHS_Woman_21_Odisha/AHS_Woman_21_Odisha.csv
Reading: ../archive/AHS_Woman_22_Chhattisgarh/AHS_Woman_22_Chhattisgarh.csv
Reading: ../archive/AHS_Woman_23_Madhya_Pradesh/AHS_Woman_23_Madhya_Pradesh_Part_1.csv
Reading: ../archive/AHS_Woman_23_Madhya_Pradesh/AHS_Woman_23_Madhya_Pradesh_Part_2.csv
Data merged
Dataset shape: (8299069, 202)


## Первичный анализ структуры данных

После загрузки данных проводится базовая разведочная проверка (EDA):

1. Вывод списка всех колонок датасета.
2. Просмотр первых строк таблицы (`head()`).
3. Анализ распределения целевой переменной `outcome_pregnancy`.
4. Проверка количества пропущенных значений в каждом столбце.

Этот этап позволяет понять структуру данных и выявить потенциальные проблемы, такие как пропуски или дисбаланс классов.

In [3]:
print("\nColumns:")
print(df.columns)

print("\nFirst rows:")
print(df.head())

print("\nOutcome distribution:")
print(df["outcome_pregnancy"].value_counts())

print("\nMissing values per column:")
print(df.isnull().sum())

print("\nData types:")
print(df.dtypes)



Columns:
Index(['w_id', 'hl_id', 'client_w_id', 'state', 'district', 'rural',
       'stratum_code', 'psu_id', 'house_no', 'house_hold_no',
       ...
       'anym', 'respondentname', 'rtelephoneno', 'isnewrecord',
       'recordupdatedcount', 'recordstatus', 'schedule_id', 'year', 'id',
       'v204'],
      dtype='object', length=202)

First rows:
   w_id hl_id  client_w_id  state  district  rural  stratum_code     psu_id  \
0   NaN   NaN          NaN      8        21      1             1  101431218   
1   NaN   NaN          NaN      8        21      1             1  101431169   
2   NaN   NaN          NaN      8        21      1             1  101431095   
3   NaN   NaN          NaN      8        21      1             1  101431064   
4   NaN   NaN          NaN      8        21      1             1  101431121   

   house_no  house_hold_no  ...  anym  respondentname  rtelephoneno  \
0        24              2  ...   NaN             NaN           NaN   
1        25              1  ..

## Очистка и подготовка целевой переменной

Целевая переменная `outcome_pregnancy` приводится к числовому типу данных.

Далее выполняется очистка выборки:
- удаляются строки с отсутствующим значением целевой переменной;
- оставляются только наблюдения с корректными значениями классов (1 и 2).

Таким образом формируется корректная выборка для задачи классификации.

In [4]:
df["outcome_pregnancy"] = pd.to_numeric(df["outcome_pregnancy"], errors="coerce")

df = df.dropna(subset=["outcome_pregnancy"])

df = df[df["outcome_pregnancy"].isin([1, 2])]

print("Cleaned target distribution:")
print(df["outcome_pregnancy"].value_counts())

Cleaned target distribution:
outcome_pregnancy
2.0    6118029
1.0    1323024
Name: count, dtype: int64


## Удаление признаков с большим количеством пропусков

Для каждого признака рассчитывается доля пропущенных значений.

Если доля пропусков превышает **90%**, такой признак считается нерелевантным и удаляется из датасета.

Это позволяет:
- уменьшить размерность данных;
- избавиться от признаков, которые не содержат полезной информации.

In [5]:
missing_ratio = df.isnull().mean()

cols_to_drop = missing_ratio[missing_ratio > 0.9].index

print("Columns to drop:", len(cols_to_drop))

df = df.drop(columns=cols_to_drop)

print("Dataset shape after dropping columns:", df.shape)

Columns to drop: 33
Dataset shape after dropping columns: (7441053, 169)


## Удаление служебных идентификаторов

Из датасета удаляются столбцы, содержащие уникальные идентификаторы записей:

- `w_id`
- `hl_id`
- `client_w_id`
- `id`
- `psu_id`

Такие признаки не несут информации для прогнозирования и могут ухудшать качество модели, поэтому они исключаются из анализа.

In [6]:
id_cols = [
    "w_id",
    "hl_id",
    "client_w_id",
    "id",
    "psu_id"
]

df = df.drop(columns=[c for c in id_cols if c in df.columns])

print("Dataset shape after removing IDs:", df.shape)

Dataset shape after removing IDs: (7441053, 164)


## Снижение размера датасета

Для ускорения обучения модели из исходного набора данных выбирается подвыборка размером **50 000 наблюдений**.

Используется стратифицированная выборка (`stratify`), чтобы сохранить исходное распределение классов целевой переменной.

Это позволяет уменьшить вычислительную нагрузку без значительной потери информации.

In [7]:
df, _ = train_test_split(
    df,
    train_size=50000,
    stratify=df["outcome_pregnancy"],
    random_state=42
)

print("Dataset size after sampling:", len(df))

Dataset size after sampling: 50000


## Определение числовых и категориальных признаков

На данном этапе признаки разделяются на две группы:

- **Числовые признаки** (`num_cols`)
- **Категориальные признаки** (`cat_cols`)

Целевая переменная `outcome_pregnancy` исключается из списка признаков.

Такое разделение необходимо для дальнейшей корректной обработки пропусков и кодирования категориальных данных.

In [8]:
num_cols = df.select_dtypes(include=["int64", "float64"]).columns
cat_cols = df.select_dtypes(include=["object"]).columns

if "outcome_pregnancy" in num_cols:
    num_cols = num_cols.drop("outcome_pregnancy")

print("Numeric columns:", len(num_cols))
print("Categorical columns:", len(cat_cols))

Numeric columns: 23
Categorical columns: 140


## Заполнение пропущенных значений

Пропуски обрабатываются отдельно для разных типов данных:

- для **числовых признаков** пропущенные значения заменяются медианой;
- для **категориальных признаков** используется специальная категория `"Unknown"`.

После выполнения операции проверяется, что в датасете больше нет пропущенных значений.

In [9]:
df[num_cols] = df[num_cols].fillna(df[num_cols].median(numeric_only=True))

df[cat_cols] = df[cat_cols].fillna("Unknown")

print("Missing values after cleaning:", df.isnull().sum().sum())

Missing values after cleaning: 0


## Формирование матрицы признаков и целевой переменной

Датасет разделяется на:

- **X** — матрицу признаков
- **y** — целевую переменную `outcome_pregnancy`

Данный формат является стандартным для обучения моделей в библиотеке scikit-learn.

In [10]:
X = df.drop(columns=["outcome_pregnancy"])
y = df["outcome_pregnancy"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (50000, 163)
y shape: (50000,)


## Разделение данных на обучающую и тестовую выборки

Данные разделяются в пропорции **80/20**:

- **обучающая выборка (train)** — используется для обучения модели;
- **тестовая выборка (test)** — используется для оценки качества модели.

Разделение выполняется стратифицированно по целевой переменной, чтобы сохранить баланс классов.

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 40000
Test size: 10000


## Кодирование категориальных признаков

Поскольку модель Random Forest работает только с числовыми данными, категориальные признаки необходимо преобразовать.

Сначала значения приводятся к строковому типу, после чего применяется **OrdinalEncoder**, который переводит категории в числовые коды.

In [13]:
# Приведение всех категориальных колонок к строкам
X_train[cat_cols] = X_train[cat_cols].astype(str)
X_test[cat_cols] = X_test[cat_cols].astype(str)

# Теперь OrdinalEncoder не будет ломаться
encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

X_train[cat_cols] = encoder.fit_transform(X_train[cat_cols])
X_test[cat_cols] = encoder.transform(X_test[cat_cols])

## Оптимизация типов данных

После кодирования все признаки приводятся к типу `float32`.

Это позволяет:
- уменьшить потребление памяти
- ускорить вычисления при обучении модели.

In [14]:
X_train = X_train.astype("float32")
X_test = X_test.astype("float32")

## Обучение модели

Для решения задачи классификации используется алгоритм **Random Forest**.

Основные параметры модели:
- `n_estimators = 100` — количество деревьев
- `max_depth = 10` — максимальная глубина деревьев
- `class_weight = "balanced"` — компенсация дисбаланса классов
- `n_jobs = -1` — использование всех доступных ядер процессора

После задания параметров модель обучается на обучающей выборке.

In [15]:
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

model.fit(X_train, y_train)

print("Model trained")

Model trained


## Получение предсказаний модели

После обучения модель применяется к тестовой выборке `X_test`.

Результатом является вектор предсказанных классов `y_pred`.

In [16]:
y_pred = model.predict(X_test)

## Оценка качества модели

Для оценки работы модели используются следующие метрики:

- **Accuracy** — доля правильных предсказаний
- **Classification Report**, который включает:
  - precision
  - recall
  - F1-score
  - support

Эти метрики позволяют оценить качество классификации для каждого класса.

In [17]:
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9656

Classification report:
              precision    recall  f1-score   support

         1.0       0.89      0.92      0.91      1778
         2.0       0.98      0.97      0.98      8222

    accuracy                           0.97     10000
   macro avg       0.94      0.95      0.94     10000
weighted avg       0.97      0.97      0.97     10000

